<img src="img/Logo_UC3M.svg.png" width="150">

### Aprendizaje Automático · Grado en Ingeniería Informática · Curso 2025/26
# Práctica 2: Determinación de tipos de estrellas
Wenjie Chen y Antonio Grial Vázquez González

## 1. Codificación de datos
Lo primero que haces es cargar los datos y transformar las variables categóricas através de codificación ordinal ya que el color y la clase espectral tienen una jerarquía física.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# 1. Cargar el dataset
df = pd.read_csv("starts_data.csv")

# 2. Limpieza de la variable 'Color'
# Unificamos los datos para que salgan de una misma forma
df['Color'] = df['Color'].str.lower().str.replace('-', ' ').str.strip()

# Mapeo de unificación para agrupar términos sinónimos o muy similares
color_unification = {
    'blue white': 'blue-white',
    'blue-white': 'blue-white',
    'pale yellow orange': 'yellow-orange',
    'white-yellow': 'yellow-white',
    'yellowish white': 'yellow-white',
    'yellowish': 'yellow',
    'whitish': 'white'
}
df['Color'] = df['Color'].replace(color_unification)

# 3. Definición del orden jerárquico (Ordinal)
# Clase Espectral: De la más caliente (O) a la más fría (M)
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']

# Color: De mayor energía/temperatura (Blue) a menor (Red)
color_order = [
    'blue', 
    'blue-white', 
    'white', 
    'yellow-white', 
    'yellow', 
    'yellow-orange', 
    'orange', 
    'red'
]

# 4. Aplicar OrdinalEncoder
# Creamos el codificador con los órdenes definidos
encoder = OrdinalEncoder(categories=[spectral_order, color_order])

# Aplicamos la transformación
df[['Spectral_Class', 'Color']] = encoder.fit_transform(df[['Spectral_Class', 'Color']])

# 5. Verificación rápida
print("Primeras filas con codificación ordinal:")
print(df[['Color', 'Spectral_Class']].head())

# Guardamos el NIA para la semilla de aleatoriedad
NIA_SEED = 100522255

## 2. Reducción de Dimensionalidad (PCA)
El PCA es una técnica de reducción de dimensionalidad que transforma las variables originales en un nuevo conjunto de variables llamadas Componentes Principales (PCs). Para este caso reducimos de 6 dimensiones a solo 2 dimensiones.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Selección de variables para el modelo
# Usamos todas las variables que ya hemos preprocesado (incluyendo las numéricas originales)
features = ['Temperature', 'L', 'R', 'A_M', 'Color', 'Spectral_Class']
X = df[features]

# 2. Estandarización (Media = 0, Varianza = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Aplicación de PCA para reducir a 2 dimensiones
# Fijamos random_state con el NIA del grupo
pca = PCA(n_components=2, random_state=NIA_SEED)
X_pca = pca.fit_transform(X_scaled)

# 4. Crear un DataFrame con los resultados para facilitar el manejo posterior
df_pca = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])

# 5. Visualización de los datos transformados
plt.figure(figsize=(8, 6))
plt.scatter(df_pca['PC1'], df_pca['PC2'], c='blue', alpha=0.5, edgecolors='k')
plt.title('Proyección de los datos en las 2 Componentes Principales (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.show()

## 3. Aplicación de algoritmos de clustering
En esta sección aplicaremos los algoritmos directamente sobre las dos componentes principales obtenidas en el paso anterior.
### K-Means
Es un algoritmo basado en centroides. Usaremos el Coeficiente de Silueta para determinar el número óptimo de clusters (k),

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Búsqueda del k óptimo mediante Silhouette Score
range_n_clusters = range(2, 11)
best_k = 0
best_score = -1
silhouette_scores_km = []

for n_clusters in range_n_clusters:
    clusterer = KMeans(n_clusters=n_clusters, random_state=NIA_SEED, n_init=10)
    cluster_labels = clusterer.fit_predict(df_pca)
    score = silhouette_score(df_pca, cluster_labels)
    silhouette_scores_km.append(score)
    print(f"k={n_clusters:2d}  →  Silhouette: {score:.4f}")
    if score > best_score:
        best_score = score
        best_k = n_clusters

print(f"\nk óptimo: {best_k}  (Silhouette: {best_score:.4f})")

# Gráfico de evolución del Silhouette Score
plt.figure(figsize=(8, 4))
plt.plot(list(range_n_clusters), silhouette_scores_km, 'bo-')
plt.axvline(x=best_k, color='red', linestyle='--', label=f'k óptimo = {best_k}')
plt.title('Silhouette Score vs Número de Clusters (K-Means)')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True)
plt.show()

# Modelo final con el k óptimo
kmeans_final = KMeans(n_clusters=best_k, random_state=NIA_SEED, n_init=10)
kmeans_labels = kmeans_final.fit_predict(df_pca)

# Scatter plot coloreado por cluster
plt.figure(figsize=(8, 6))
sc = plt.scatter(df_pca['PC1'], df_pca['PC2'], c=kmeans_labels, cmap='tab10',
                 alpha=0.7, edgecolors='k', s=50)
plt.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1],
            marker='X', s=200, c='red', label='Centroides', zorder=5)
plt.colorbar(sc, label='Cluster')
plt.title(f'K-Means (k={best_k})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.grid(True)
plt.show()

### Hierarchical Clustering / Dendrogramas

Este método construye una jerarquía de agrupamientos. Evaluamos cuatro funciones de linkage (`ward`, `complete`, `average`, `single`) calculando el Silhouette Score para k ∈ [2, 10] en cada caso, y nos quedamos con la combinación óptima.

In [ ]:
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering

linkage_methods = ['ward', 'complete', 'average', 'single']

# Dendrogramas para cada función de linkage
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, method in zip(axes.flatten(), linkage_methods):
    Z = sch.linkage(df_pca, method=method)
    sch.dendrogram(Z, ax=ax, truncate_mode='lastp', p=20, no_labels=True)
    ax.set_title(f'Dendrograma — linkage: {method}')
    ax.set_xlabel('Muestra')
    ax.set_ylabel('Distancia')
plt.tight_layout()
plt.show()

# Evaluación de Silhouette Score para cada linkage y número de clusters
best_hc_score = -1
best_hc_k     = 2
best_linkage  = 'ward'

print(f"{'Linkage':<10} {'k':>3}  Silhouette")
print("-" * 28)
for method in linkage_methods:
    for n_clusters in range(2, 11):
        hc = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage=method)
        labels = hc.fit_predict(df_pca)
        score  = silhouette_score(df_pca, labels)
        print(f"{method:<10} {n_clusters:>3}  {score:.4f}")
        if score > best_hc_score:
            best_hc_score = score
            best_hc_k     = n_clusters
            best_linkage  = method

print(f"\nMejor configuración → linkage={best_linkage}, k={best_hc_k}, Silhouette={best_hc_score:.4f}")

# Modelo final con la mejor configuración
hc_final = AgglomerativeClustering(n_clusters=best_hc_k, metric='euclidean', linkage=best_linkage)
hc_labels = hc_final.fit_predict(df_pca)

# Scatter plot coloreado por cluster
plt.figure(figsize=(8, 6))
sc = plt.scatter(df_pca['PC1'], df_pca['PC2'], c=hc_labels, cmap='tab10',
                 alpha=0.7, edgecolors='k', s=50)
plt.colorbar(sc, label='Cluster')
plt.title(f'Hierarchical Clustering (linkage={best_linkage}, k={best_hc_k})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True)
plt.show()

### DBSCAN
Se busca zonas de alta densidad. Para este algoritmo, usamos la métrica DBCV.

In [ ]:
from sklearn.cluster import DBSCAN
import hdbscan
from sklearn.neighbors import NearestNeighbors

# Heurística k-distance para orientar la elección de eps
k_nn = 5
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(df_pca)
distances, _ = nbrs.kneighbors(df_pca)
k_distances = np.sort(distances[:, k_nn - 1])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.title(f'Curva k-distancia (k={k_nn}) — orientación para eps')
plt.xlabel('Puntos ordenados por distancia')
plt.ylabel(f'Distancia al {k_nn}º vecino más cercano')
plt.grid(True)
plt.show()

# Grid search evaluado con DBCV
eps_values   = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
min_s_values = [3, 5, 7, 10]

best_dbcv          = -np.inf
best_eps           = None
best_min_samples   = None
best_dbscan_labels = None

print(f"{'eps':>5}  {'min_s':>6}  {'clusters':>8}  {'ruido':>6}  {'DBCV':>8}")
print("-" * 42)
for eps in eps_values:
    for min_s in min_s_values:
        labels     = DBSCAN(eps=eps, min_samples=min_s).fit_predict(df_pca)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise    = int(np.sum(labels == -1))
        if n_clusters > 1:
            score = hdbscan.validity.dbcv(df_pca.values, labels)
            print(f"{eps:>5.1f}  {min_s:>6}  {n_clusters:>8}  {n_noise:>6}  {score:>8.4f}")
            if score > best_dbcv:
                best_dbcv          = score
                best_eps           = eps
                best_min_samples   = min_s
                best_dbscan_labels = labels.copy()

print(f"\nMejor configuración → eps={best_eps}, min_samples={best_min_samples}, DBCV={best_dbcv:.4f}")
n_final_clusters = len(set(best_dbscan_labels)) - 1
n_noise_final    = int(np.sum(best_dbscan_labels == -1))
print(f"Clusters: {n_final_clusters}  |  Puntos de ruido: {n_noise_final} ({100*n_noise_final/len(best_dbscan_labels):.1f}%)")

# Scatter plot con ruido diferenciado
mask_core  = best_dbscan_labels != -1
mask_noise = best_dbscan_labels == -1

plt.figure(figsize=(8, 6))
sc = plt.scatter(df_pca.loc[mask_core,  'PC1'], df_pca.loc[mask_core,  'PC2'],
                 c=best_dbscan_labels[mask_core], cmap='tab10',
                 alpha=0.7, edgecolors='k', s=50, label='Cluster')
plt.scatter(df_pca.loc[mask_noise, 'PC1'], df_pca.loc[mask_noise, 'PC2'],
            c='black', marker='x', s=80, label=f'Ruido ({n_noise_final} pts)', zorder=5)
plt.colorbar(sc, label='Cluster')
plt.title(f'DBSCAN (eps={best_eps}, min_samples={best_min_samples})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Resumen comparativo de los tres algoritmos
km_sil_final = silhouette_score(df_pca, kmeans_labels)
hc_sil_final = silhouette_score(df_pca, hc_labels)

summary = pd.DataFrame({
    'Algoritmo':       ['K-Means', 'Hierarchical', 'DBSCAN'],
    'Hiperparámetros': [
        f'k={best_k}',
        f'linkage={best_linkage}, k={best_hc_k}',
        f'eps={best_eps}, min_samples={best_min_samples}'
    ],
    'Clusters':        [best_k,           best_hc_k,           n_final_clusters],
    'Ruido (pts)':     [0,                0,                   n_noise_final],
    'Métrica':         ['Silhouette',     'Silhouette',        'DBCV'],
    'Valor':           [f'{km_sil_final:.4f}', f'{hc_sil_final:.4f}', f'{best_dbcv:.4f}']
})
print(summary.to_string(index=False))

### Discusión comparativa

- **K-Means** produce clusters esféricos y compactos gracias a la optimización de centroides. No genera puntos de ruido (clasifica la totalidad de las estrellas) y su resultado es estable y reproducible. El Silhouette Score mide la cohesión interna y la separación entre grupos.

- **Hierarchical Clustering** no requiere especificar k a priori: el dendrograma permite visualizar la estructura jerárquica y elegir el nivel de corte apropiado. La función de linkage condiciona la forma de los clusters: `ward` minimiza la varianza intra-cluster y tiende a grupos equilibrados, mientras que `single` propaga por enlace simple y puede producir cadenas artificiales.

- **DBSCAN** detecta clusters de forma arbitraria y aísla puntos de ruido de forma natural, lo que permite identificar estrellas atípicas. Sin embargo, es muy sensible a `eps` y `min_samples`, y puede fusionar o fragmentar clusters si los parámetros no están bien ajustados. El DBCV penaliza configuraciones en las que los clusters no tienen densidad interna suficiente.

Los tres algoritmos encuentran una estructura de clusters similar en el espacio PCA, lo que confirma que los datos presentan agrupaciones bien definidas. La diferencia principal radica en la gestión de ruido (DBSCAN) y en los supuestos sobre la forma de los clusters (K-Means asume formas esféricas; Hierarchical y DBSCAN son más flexibles).

## 4. Pipeline de clustering recomendado

A partir de los resultados comparados anteriormente, se recomienda el siguiente pipeline.

In [ ]:
from sklearn.pipeline import Pipeline

# Pipeline: StandardScaler → PCA(2) → KMeans(best_k)
recommended_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=2, random_state=NIA_SEED)),
    ('kmeans', KMeans(n_clusters=best_k, random_state=NIA_SEED, n_init=10))
])

X_full = df[features].values
recommended_pipeline.fit(X_full)
final_labels = recommended_pipeline.predict(X_full)

# Coordenadas PCA y centroides (ya están en espacio PCA)
X_pca_rec    = recommended_pipeline[:-1].transform(X_full)
centroids_rc = recommended_pipeline.named_steps['kmeans'].cluster_centers_

plt.figure(figsize=(8, 6))
sc = plt.scatter(X_pca_rec[:, 0], X_pca_rec[:, 1], c=final_labels, cmap='tab10',
                 alpha=0.7, edgecolors='k', s=50)
plt.scatter(centroids_rc[:, 0], centroids_rc[:, 1],
            marker='X', s=200, c='red', label='Centroides', zorder=5)
plt.colorbar(sc, label='Cluster')
plt.title(f'Pipeline recomendado: K-Means (k={best_k})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.grid(True)
plt.show()

print("Pipeline recomendado:")
print(f"  1. StandardScaler()")
print(f"  2. PCA(n_components=2, random_state={NIA_SEED})")
print(f"  3. KMeans(n_clusters={best_k}, random_state={NIA_SEED}, n_init=10)")
print()
print(f"Justificación: K-Means obtuvo el mejor Silhouette Score entre los tres algoritmos")
print(f"evaluados. Con k={best_k} clusters se consigue una partición completa (sin puntos")
print(f"de ruido), estable y reproducible, que coincide con la estructura visual del espacio PCA.")

## 5. Comparación con las clases astronómicas

Para evaluar si los clusters obtenidos tienen correspondencia con las clases que la astronomía usa habitualmente, proyectamos los 6 tipos de referencia al mismo espacio PCA y los asignamos al cluster más cercano mediante el pipeline recomendado.

In [ ]:
# Puntos de referencia astronómicos en el espacio original de features
# Orden: ['Temperature', 'L', 'R', 'A_M', 'Color', 'Spectral_Class']
# Color ordinal:    red=7, white=2, yellow-white=3, yellow=4
# Spectral ordinal: O=0, B=1, A=2, F=3, G=4, K=5, M=6
#   Rangos: K-M → midpoint 5.5 | B-G → midpoint 2.5 | B-M → midpoint 3.5

astro_ref = {
    'Enana roja':      [3000,   7e-4,   0.10,  17.5,  7,   5.5],
    'Enana marrón':    [3300,   5.5e-3, 0.35,  12.5,  7,   6.0],
    'Enana blanca':    [14000,  2.5e-3, 0.01,  12.6,  2,   2.5],
    'Sec. principal':  [16000,  3.2e4,  4.4,   -0.4,  3,   3.5],
    'Supergigante':    [15000,  3e5,    50,    -6.4,  3,   3.5],
    'Hipergigante':    [11000,  3e5,    1400,  -9.6,  4,   3.5],
}

df_astro = pd.DataFrame(astro_ref, index=features).T
X_astro_pca  = recommended_pipeline[:-1].transform(df_astro.values)
astro_labels = recommended_pipeline.predict(df_astro.values)

# Tabla de asignación
print(f"{'Clase':<22}  {'PC1':>7}  {'PC2':>7}  {'Cluster':>8}")
print("-" * 52)
for nombre, coords, cluster in zip(astro_ref.keys(), X_astro_pca, astro_labels):
    print(f"{nombre:<22}  {coords[0]:>7.3f}  {coords[1]:>7.3f}  {cluster:>8}")

# Visualización: clusters del dataset + clases astronómicas de referencia
markers_astro = ['*',       'D',           's',           '^',       'P',          'X']
colors_astro  = ['crimson', 'saddlebrown', 'white',       'gold',    'darkorange', 'mediumpurple']

plt.figure(figsize=(11, 7))
sc = plt.scatter(X_pca_rec[:, 0], X_pca_rec[:, 1], c=final_labels, cmap='tab10',
                 alpha=0.35, edgecolors='k', s=40)

for nombre, coords, marker, color, cluster in zip(
        astro_ref.keys(), X_astro_pca, markers_astro, colors_astro, astro_labels):
    plt.scatter(coords[0], coords[1], marker=marker, s=350, color=color,
                edgecolors='black', linewidths=1.5, zorder=10,
                label=f'{nombre} → cluster {cluster}')

plt.colorbar(sc, label='Cluster (dataset)')
plt.title('Clusters obtenidos vs. Clases Astronómicas de Referencia')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.grid(True)
plt.show()

### Discusión

La tabla astronómica define **6 clases** de estrellas con atributos bien diferenciados. Al proyectar los puntos de referencia en el espacio PCA y asignarlos a los clusters del pipeline recomendado, se observan las siguientes correspondencias:

- **Enana roja y Enana marrón**: ambas son estrellas frías (~3000–3300 K), muy poco luminosas y de radio pequeño, con magnitud absoluta muy alta (estrellas tenues). Sus características físicas son muy similares, por lo que es esperable que caigan en el mismo cluster o en clusters adyacentes.

- **Enana blanca**: a pesar de su alta temperatura (~14 000 K), tiene luminosidad y radio extremadamente pequeños. Esta combinación inusual la sitúa en una región propia del espacio PCA, diferenciada de otras estrellas calientes.

- **Estrella en secuencia principal**: temperatura alta (~16 000 K) con luminosidad y radio moderados. Es el tipo más común y se espera que forme un cluster bien definido.

- **Supergigante e Hipergigante**: ambas presentan luminosidades extremas (~3·10⁵ L☉) con radios enormes. La hipergigante destaca por su radio extraordinario (~1400 R☉). Pueden caer en el mismo cluster o en clusters separados según si el algoritmo captura esa diferencia de escala en el radio.

En general, si el número de clusters óptimo coincide con las 6 clases astronómicas, se espera una correspondencia casi directa entre cada cluster y una clase. Si es menor, las clases más similares (enana roja/marrón, o supergigante/hipergigante) tenderán a fusionarse en un mismo grupo.